# Hinglish SentiMix — GPU runner (Colab / Kaggle)
Runtime → Change runtime type → **T4 GPU**. Everything is resumable: if Colab disconnects, re-run all cells and finished runs are skipped.

Results are written inside the repo folder on Google Drive so nothing is lost.

In [ ]:
# 1. Mount Drive + get the repo (edit REPO_URL)
from google.colab import drive
drive.mount('/content/drive')
REPO_URL = 'https://github.com/Raajp10/hinglish-sentimix.git'  # <-- your repo
import os
WORK = '/content/drive/MyDrive/hinglish-sentimix'
if not os.path.exists(WORK):
    !git clone $REPO_URL $WORK
else:
    !cd $WORK && git pull
%cd $WORK

In [ ]:
!pip install -q -r requirements.txt
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 2. Data + normalization dict + baseline (CPU, ~2 min)
!python -m src.prepare > /dev/null && python -m src.normalize | head -1
!python -m src.baseline | tail -1
!python -m src.baseline --normalize | tail -1

## 3. Transformers
Per model: LR sweep (4 LRs × 4 epochs, seed 42, val only) then 3 seeds on test with the best (lr, epochs). ~35–45 min per model on a T4.

Run one model per cell so a crash only costs that model. Fast mode: add `--lrs 2e-5 3e-5`.

In [ ]:
!python -m src.sweep --models mbert --save_model

In [ ]:
!python -m src.sweep --models xlmr --save_model

In [ ]:
!python -m src.sweep --models indicbert --save_model

In [ ]:
!python -m src.sweep --models hingbert --save_model

In [ ]:
# 4. Which transformer is weakest? -> run the normalization experiment on it
import pandas as pd, subprocess
subprocess.run(['python','-m','src.analyze'], check=True, capture_output=True)
df = pd.read_csv('results/tables/main_results.csv')
weak = df[~df.tag.str.contains('tfidf|\\+norm')].sort_values('weighted_f1').tag.iloc[0]
print('weakest transformer:', weak)
!python -m src.sweep --models $weak --stage final --norm_models $weak

In [ ]:
# 5. Tables, figures, error-review sheet
!python -m src.analyze
print(open('results/tables/main_results.md').read())

In [ ]:
from IPython.display import Image, display
for f in ['main_results','f1_by_cmi','f1_by_length','confusion_matrices']:
    display(Image(f'results/figures/{f}.png', width=700))

In [ ]:
# 6. Optional demo (public link)
!python -m demo.app --share